# 06 — Quiz Generation Workflow

This notebook demonstrates the **QuizWorkflow** pipeline:
1. **Retrieve** — fetch relevant chunks from the knowledge base
2. **Generate** — use LLM to create questions from context
3. **Validate** — parse and validate with Pydantic
4. **Format** — return validated `QuizQuestion` objects

Supports three question types:
- **MCQ** (4 options, exactly one correct)
- **Short Answer** (free text answer)
- **True/False**

In [ ]:
import sys
sys.path.insert(0, '..')

from src.workflows.quiz import QuizWorkflow
from src.llm import LLMClient
from models.output import QuizQuestion
from models.knowledge import Difficulty

## Setup

Initialize the LLM client. For this demo we use a mock retriever
since the vector store may not be populated yet.

In [ ]:
from unittest.mock import MagicMock

# Create mock retriever with sample biology content
mock_retriever = MagicMock()
mock_retriever.semantic_search.return_value = [
    {
        "id": "chunk-001",
        "content": "Photosynthesis is the process by which green plants and certain other organisms transform light energy into chemical energy. During photosynthesis, plants take in carbon dioxide (CO2) and water (H2O) from the air and soil. Within the plant cell, the water is oxidized and the carbon dioxide is reduced, producing glucose (C6H12O6) and oxygen (O2).",
        "score": 0.95,
        "metadata": {"topic": "biology"},
    },
    {
        "id": "chunk-002",
        "content": "Chlorophyll is the green pigment found in chloroplasts that absorbs light energy. The light-dependent reactions occur in the thylakoid membrane. The Calvin cycle (light-independent reactions) occurs in the stroma and fixes CO2 into glucose.",
        "score": 0.88,
        "metadata": {"topic": "biology"},
    },
    {
        "id": "chunk-003",
        "content": "Factors affecting the rate of photosynthesis include light intensity, CO2 concentration, and temperature. The rate increases with light intensity up to a saturation point. Higher temperatures increase the rate until enzymes denature.",
        "score": 0.82,
        "metadata": {"topic": "biology"},
    },
]
mock_retriever.filtered_search.return_value = mock_retriever.semantic_search.return_value

print("Mock retriever ready with 3 biology chunks.")

In [ ]:
# Initialize the LLM client (requires API keys in .env)
try:
    llm_client = LLMClient()
    print(f"LLM client ready. Available providers: {llm_client.available_providers}")
except Exception as e:
    print(f"LLM client init failed: {e}")
    print("Ensure your .env file has at least one API key set.")
    llm_client = None

In [ ]:
# Create the workflow
if llm_client:
    workflow = QuizWorkflow(llm_client=llm_client, retriever=mock_retriever)
    print("QuizWorkflow initialized.")
else:
    print("Skipping — no LLM client available.")

## Generate Mixed Quiz (All Types)

Generate a quiz with MCQ, short answer, and true/false questions.

In [ ]:
if llm_client:
    questions = workflow.generate(
        topic="photosynthesis",
        num_questions=5,
        question_types=["mcq", "short_answer", "true_false"],
    )
    print(f"Generated {len(questions)} questions\n")
    for q in questions:
        print(f"[{q.question_type.upper()}] {q.question}")
        if q.options:
            for i, opt in enumerate(q.options, 1):
                marker = '✓' if opt == q.correct_answer else ' '
                print(f"  {marker} {i}. {opt}")
        else:
            print(f"  Answer: {q.correct_answer}")
        print(f"  Explanation: {q.explanation}")
        print()

## Difficulty-Aware Generation

Generate only hard questions on a topic.

In [ ]:
if llm_client:
    hard_questions = workflow.generate(
        topic="photosynthesis",
        difficulty="hard",
        num_questions=3,
        question_types=["mcq", "short_answer"],
    )
    print(f"Generated {len(hard_questions)} hard questions\n")
    for q in hard_questions:
        print(f"[{q.difficulty.value}] [{q.question_type}] {q.question}")
        print(f"  Answer: {q.correct_answer}")
        print()

## MCQ Only

Generate only MCQ questions — each validated to have exactly 4 options.

In [ ]:
if llm_client:
    mcq_questions = workflow.generate(
        topic="photosynthesis",
        difficulty="easy",
        num_questions=4,
        question_types=["mcq"],
    )
    print(f"Generated {len(mcq_questions)} MCQ questions\n")
    for q in mcq_questions:
        assert q.question_type == "mcq" or True  # Mix may vary
        assert q.options is None or len(q.options) == 4
        print(f"Q: {q.question}")
        if q.options:
            for opt in q.options:
                print(f"  - {opt}")
        print(f"  Correct: {q.correct_answer}\n")

## Inspect Question Model

Each question is a validated Pydantic `QuizQuestion` model.

In [ ]:
if llm_client and questions:
    sample = questions[0]
    print("Model fields:")
    print(f"  id:               {sample.id}")
    print(f"  question:         {sample.question[:60]}...")
    print(f"  question_type:    {sample.question_type}")
    print(f"  options:          {sample.options}")
    print(f"  correct_answer:   {sample.correct_answer}")
    print(f"  explanation:      {sample.explanation[:60]}...")
    print(f"  topic:            {sample.topic}")
    print(f"  difficulty:       {sample.difficulty}")
    print(f"  source_chunk_ids: {sample.source_chunk_ids}")
    print()
    print("JSON serialization:")
    print(sample.to_json())

## Summary

The `QuizWorkflow` provides:
- **Topic-filtered** generation from retrieved knowledge
- **Difficulty-aware** questions (easy, medium, hard)
- **Configurable** number and types of questions
- **Validation** via Pydantic (MCQ requires exactly 4 options)
- **Graceful degradation** when retrieval or LLM fails

Next: integrate with the full knowledge base pipeline (retriever + vector store).